In [12]:
# Recreate PDF documents and chunks

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "/content/2005.11401v4.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Pages:", len(documents))
print("Chunks:", len(chunks))

Pages: 19
Chunks: 92


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings ready!


In [14]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rag_document"
)

print("Vector database created successfully!")

Vector database created successfully!


In [15]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Retriever ready!")

Retriever ready!


In [16]:
query = "What is Retrieval-Augmented Generation?"

results = retriever.invoke(query)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content[:1000])


--- Result 1 ---
in 71% of cases, and a gold article is present in the top 10 retrieved articles in 90% of cases.
4.5 Additional Results
Generation Diversity Section 4.3 shows that RAG models are more factual and speciﬁc than
BART for Jeopardy question generation. Following recent work on diversity-promoting decoding
[33, 59, 39], we also investigate generation diversity by calculating the ratio of distinct ngrams to
total ngrams generated by different models. Table 5 shows that RAG-Sequence’s generations are
more diverse than RAG-Token’s, and both are signiﬁcantly more diverse than BART without needing
any diversity-promoting decoding.
Retrieval Ablations A key feature of RAG is learning to retrieve relevant information for the task.
To assess the effectiveness of the retrieval mechanism, we run ablations where we freeze the retriever
during training. As shown in Table 6, learned retrieval improves results for all tasks.

--- Result 2 ---
could represent promising future work.
6 Disc

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("LLM loaded successfully!")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully!


In [19]:
query = "What is Retrieval-Augmented Generation?"

results = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in results])

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
)

outputs = model.generate(
    **inputs,
    max_new_tokens=150
)

answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Question:", query)
print("\nAnswer:", answer)

Question: What is Retrieval-Augmented Generation?

Answer: Learning to retrieve relevant information for the task


In [21]:
questions = [
    "What is Retrieval-Augmented Generation?",
    "Why is retrieval important in RAG?",
    "What are the main components of a RAG system?"
]

for query in questions:
    results = retriever.invoke(query)
    context = "\n\n".join([doc.page_content for doc in results])

    prompt = f"""
    Answer the question using only the provided context.
    If the answer is not available in the context, say "Not found in the document."

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    )

    outputs = model.generate(
        **inputs,
        max_new_tokens=150
    )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("=" * 70)
    print("QUESTION:", query)
    print("ANSWER:", answer)

QUESTION: What is Retrieval-Augmented Generation?
ANSWER: Learning to retrieve relevant information for the task
QUESTION: Why is retrieval important in RAG?
ANSWER: Learning to retrieve relevant information for the task
QUESTION: What are the main components of a RAG system?
ANSWER: G Parameters
